# Day 8 — Bagging and random forests: averaging away variance

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Setup — reuse Day 6/7's Titanic cleaning

Same 4 features as Day 7 (`pclass`, `fare`, `who`, `family_size`), same split. One addition: `X_train`/`X_test` as raw numpy arrays (`.values`), because the bagging loop below needs to index rows by integer position (`X[idx]`) to build bootstrap samples — a DataFrame would need `.iloc` for the same thing, numpy is just less ceremony here.

In [ ]:
df = sns.load_dataset("titanic")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"C": 0, "Q": 1, "S": 2})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.drop(
    columns=[
        "class",
        "embark_town",
        "alive",
        "alone",
        "sibsp",
        "parch",
        "deck",
        "adult_male",
    ],
    inplace=True,
)

median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
df["embarked"] = df["embarked"].astype(int)
df["who"] = df["who"].map({"man": 0, "woman": 1, "child": 2})
df.drop(["sex", "age", "embarked"], axis=1, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)
X_train, X_test = x_train.values, x_test.values
y_train_v, y_test_v = y_train.values, y_test.values

print("train shape:", X_train.shape, " test shape:", X_test.shape)

## Step 1 — build bagging from scratch, watch it plateau

Two ingredients, deliberately isolated from each other:

1. **Bootstrap resampling**: for each tree, draw `n` row-indices *with replacement* from the `n`-row training set (`rng.integers(0, n, n)`). Some rows get picked 2-3 times, some get skipped entirely — each tree trains on a slightly different dataset, which is the only thing making the trees different from each other (same algorithm, same `max_depth`, different data).
2. **Majority vote**: at prediction time, every tree votes (0 or 1), and the ensemble takes whichever label the majority picked. For binary labels, `mean(predictions) >= 0.5` is exactly majority vote — no separate voting library needed.

No feature subsampling yet — every tree still considers all 4 features at every split. That's tomorrow's ingredient (the "random" in random forest); today isolates just "many trees, averaged" as its own idea.

In [ ]:
def bagging_ensemble(X, y, n_trees, max_depth=4, seed=42):
    rng = np.random.default_rng(seed)
    trees = []
    n = len(y)
    for i in range(n_trees):
        idx = rng.integers(0, n, n)  # bootstrap: sample with replacement
        t = DecisionTreeClassifier(max_depth=max_depth, random_state=seed + i)
        t.fit(X[idx], y[idx])
        trees.append(t)
    return trees


def bagging_predict(trees, X):
    preds = np.array([t.predict(X) for t in trees])
    return (preds.mean(axis=0) >= 0.5).astype(int)


for n_trees in [1, 5, 10, 25, 50, 100]:
    trees = bagging_ensemble(X_train, y_train_v, n_trees=n_trees, max_depth=4, seed=42)
    preds = bagging_predict(trees, X_test)
    acc = accuracy_score(y_test_v, preds)
    print(f"n_trees={n_trees:>3}: test acc={acc:.4f}")

## Step 2 — add feature subsampling, the actual "random" in random forest

Step 1's bagging still let every tree see all 4 features at every split — with a feature this strong (`who`, importance 0.63 in Day 7), most bootstrapped trees will still split on `who` first regardless of which rows they got, so the trees end up more similar to each other than pure row-resampling alone would suggest.

`RandomForestClassifier` adds a second randomization: **at every split, only a random subset of features is even considered** (`max_features`, default `sqrt(n_features)` for classification — with 4 features here, that's 2 candidate features per split). This forces some splits to happen on `pclass` or `fare` even when `who` would've won a fair fight, which decorrelates the trees. Breiman's original argument: a forest's error depends on both how accurate individual trees are *and* how correlated their errors are with each other — decorrelating helps even when it makes individual trees slightly weaker, because the majority vote benefits more from independence than from any single tree's strength.

CV sweep over `n_estimators` (train-only, same discipline as Day 7):

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for n in [1, 5, 10, 25, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    scores = cross_val_score(rf, x_train, y_train, cv=5)
    print(f"n_estimators={n:>3}: CV acc={scores.mean():.4f} (+/-{scores.std():.4f})")

Not a clean "more trees = monotonically better/less variance" curve, and that's shown honestly rather than smoothed over. Two real reasons, not a bug:

1. **CV std here is itself an estimate of only 5 numbers** — the std-of-a-std has its own sampling noise, so small differences in the reported std (0.0098 to 0.0190) aren't necessarily meaningful.
2. **`n_estimators=1` isn't a fair stand-in for Step 1's `n_trees=1`.** Even with a single tree, `RandomForestClassifier` still bootstraps its row sample *and* subsamples features at every split — it's already a different, less-variable construction than a plain unconstrained `DecisionTreeClassifier`.

The one thing that *is* trustworthy is `x_train` never being touched by `x_test` at this stage — every number above came from CV only, exactly like every accuracy figure since Day 7.

## Step 3 — grid search, one honest evaluation

Step 2 only swept `n_estimators` at the default `max_depth=None`. Now grid-search `n_estimators × max_depth` jointly via CV (train-only, 5-fold, exactly Day 7's discipline) — 24 combinations, 5 fits each. Pick the single best combination by CV mean, retrain it once on the full `x_train`, then touch `x_test` exactly once for the final number.

In [ ]:
n_estimators_grid = [10, 50, 100, 200]
max_depth_grid = [3, 4, 5, 6, 8, None]

results = []
for n_est in n_estimators_grid:
    for depth in max_depth_grid:
        rf = RandomForestClassifier(
            n_estimators=n_est, max_depth=depth, random_state=42
        )
        scores = cross_val_score(rf, x_train, y_train, cv=5)
        results.append((n_est, depth, scores.mean(), scores.std()))

results.sort(key=lambda r: -r[2])
print("top 5 configs by CV mean:")
for n_est, depth, mean, std in results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={str(depth):<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n, best_depth, best_cv, best_std = results[0]
print(
    f"\nbest config: n_estimators={best_n}, max_depth={best_depth}, CV acc={best_cv:.4f} (+/-{best_std:.4f})"
)

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=best_n, max_depth=best_depth, random_state=42
)
best_rf.fit(x_train, y_train)

test_preds = best_rf.predict(x_test)
final_test_acc = accuracy_score(y_test, test_preds)
cm = confusion_matrix(y_test, test_preds)

print(f"Final ONE-TIME test accuracy: {final_test_acc}")
print("confusion matrix:\n", cm)

print("\nModel                              Test accuracy")
print("Day 6 logistic regression (L2)      0.8101")
print("Day 7 single tree (CV-tuned)         0.8212")
print(f"Day 8 random forest (CV-tuned)       {final_test_acc:.4f}")

**Honest reading, not the pasted lesson's story**: the pasted lesson had its CV-tuned forest as "the first model to beat the logistic regression baseline." Ours doesn't — the forest's final test accuracy (0.8212) **exactly ties** Day 7's single CV-tuned tree, and both edge out Day 6's logistic regression by the same margin. Two things worth taking from that rather than forcing a "forest wins" narrative:

- **The CV mean did go up meaningfully** (0.8370 vs. Day 7's best single-tree CV of 0.8286) — the forest's cross-validated estimate is more optimistic and, with `std=0.0218`, still fairly uncertain. That the *final test number* didn't move past Day 7's is a reminder that CV improvement doesn't guarantee a better number on any one held-out set — it's a claim about the average case, not this specific 179-row draw.
- **The identical confusion matrix** `[[92, 13], [19, 55]]` for both Day 7's tree and this forest is not a coincidence worth over-reading either — it means these two specific models happened to make exactly the same calls on exactly these 179 people. With a feature set this small (4 columns, one of which dominates), that's plausible without implying the two models are "the same" in any deeper sense.
- The real, defensible conclusion for this dataset: ensembling gave a more *stable* estimate (visible in the CV numbers), not a definitively *higher* one on this particular test split — which is itself a fair, useful lesson about what ensembling promises and doesn't.

## Step 4 — feature importance shifts

`best_rf.feature_importances_` is the same Gini-based measure as Day 7's single tree, just averaged across all 50 trees in the forest instead of read from one. Compare it directly against Day 7's `max_depth=3` tree importances.

In [ ]:
print("Random forest feature importances:")
for name, imp in sorted(
    zip(x_train.columns, best_rf.feature_importances_), key=lambda t: -t[1]
):
    print(f"  {name:<12}: {imp:.4f}")

print("\nDay 7 single tree (max_depth=3) importances, for comparison:")
day7 = {"who": 0.6277, "pclass": 0.2191, "fare": 0.1532, "family_size": 0.0000}
for name, imp in day7.items():
    print(f"  {name:<12}: {imp:.4f}")

Real, and it shows the mechanism working exactly as Step 2 described it: `family_size` goes from **zero** credit in Day 7's tree to **0.1175** here — not because it suddenly became more predictive, but because feature subsampling forces some splits, across the forest's 50 trees × up to depth 8, to happen without `who` even being a candidate. `family_size` finally gets chances to prove itself in those splits where it does. `fare` jumps from third place (0.1532) to second (0.3382) for the same reason — plus the forest's greater depth budget (8 vs. 3) gives every feature more total split opportunities across the ensemble.

`who` is still the single strongest feature in both models (0.63 → 0.42), just less dominant once it's not automatically available at every split. That's the concrete, numeric signature of decorrelation: importance spreading out isn't a side effect, it's the direct result of deliberately handicapping the strongest feature so the ensemble stops leaning on one signal.